# Verify bug fixes and new grammar (docs/issues/)

Exercises, against a live `mnemosyne_server`, the fixes for:
- `aliases_not_working.md` — `AS` alias on aggregate output (e.g. `COUNT(name) AS total_customers`)
- `limit_grammar_not_working.md` — `LIMIT n` actually limiting rows
- `overwriting_files.md` — `File` engine `INSERT` appending instead of overwriting
- `no_quartile_percentile_grammar.md` — new `QUANTILE`, `PERCENTILE` aggregates and `NTILE` window function

Run cells top to bottom. The last cell stops the server it started.

In [1]:
import pandas as pd
import sys
sys.path.append("../")
sys.path.append("../scripts")

In [2]:
import json
import subprocess
import tempfile
import time
from pathlib import Path

In [3]:
from _mnemo_client import (
    Client,
    TestRunner,
    find_server_bin,
    GREEN,
    RED,
    YELLOW,
    RESET,
)

## Launch a fresh server

Builds are expected under `build/bin/Debug` or `build/bin/Release` (see `find_server_bin`
in `scripts/_mnemo_client.py`). Rebuild first if this fails to find a binary:

```
cmake --build build --target mnemosyne_server --config Debug
```

In [4]:
server_bin = find_server_bin()
assert server_bin is not None, (
    "mnemosyne_server binary not found — build it first: "
    "cmake --build build --target mnemosyne_server --config Debug"
)

repo_root = Path("..").resolve()
server_proc = subprocess.Popen(
    [str(server_bin)],
    cwd=str(repo_root),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

client = Client("http://127.0.0.1:1143")

deadline = time.time() + 20
ready = False
while time.time() < deadline:
    if server_proc.poll() is not None:
        raise RuntimeError("mnemosyne_server exited before becoming ready")
    try:
        if client.get("/ping", timeout=1.0).status == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(0.3)

assert ready, "server did not become ready in time"
print(f"{GREEN}Server ready at {client.base_url} (pid={server_proc.pid}, bin={server_bin}){RESET}")

Server ready at http://127.0.0.1:1143 (pid=47468, bin=C:\Users\tsuma.thomas\Documents\Mnemosyne\build\bin\Release\mnemosyne_server.exe)


In [5]:
t = TestRunner()

DB = "bugfix_verification_db"
TABLE = "customers"

t.section("Setup")
r = client.query(f"CREATE DATABASE IF NOT EXISTS {DB}")
t.check("CREATE DATABASE", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(f"USE DATABASE {DB}")
t.check("USE DATABASE", r.ok(), f"status={r.status} body={r.body!r}")

r = client.query(
    f"CREATE TABLE IF NOT EXISTS {TABLE} "
    f"(customer_id Int64, name String, age Int64) ENGINE=Memory"
)
t.check("CREATE TABLE customers", r.ok(), f"status={r.status} body={r.body!r}")

# ages: Alice 28, Bob 22, Carl 27, Dana 21, Eve 35 -> sorted: 21, 22, 27, 28, 35
rows = [(1, "Alice", 28), (2, "Bob", 22), (3, "Carl", 27), (4, "Dana", 21), (5, "Eve", 35)]
for cid, name, age in rows:
    r = client.query(f"INSERT INTO {TABLE} VALUES ({cid}, '{name}', {age})")
    t.check(f"INSERT {name}", r.ok(), f"status={r.status} body={r.body!r}")


== Setup ==
  PASS CREATE DATABASE
  PASS USE DATABASE
  PASS CREATE TABLE customers
  PASS INSERT Alice
  PASS INSERT Bob
  PASS INSERT Carl
  PASS INSERT Dana
  PASS INSERT Eve


## Bug fix 1 — `AS` alias on aggregate output (`aliases_not_working.md`)

Before the fix, `SELECT COUNT(name) as total_customers FROM customers` returned a column
literally named `COUNT(name)`, ignoring the alias.

In [6]:
t.section("Bug fix 1: AS alias on aggregate output")

r = client.query(f"SELECT COUNT(name) as total_customers FROM {TABLE}")
result = json.loads(r.body)
print(result)

t.check("query succeeds", r.ok(), f"status={r.status} body={r.body!r}")
t.check(
    "column name is 'total_customers' (not 'COUNT(name)')",
    result.get("columns") == ["total_customers"],
    f"columns={result.get('columns')}",
)
t.check("count value is correct (5 rows)", result.get("data") == [[5]], f"data={result.get('data')}")


== Bug fix 1: AS alias on aggregate output ==
{'rows': 1, 'columns': ['COUNT(name)'], 'data': [[5]], 'duration_ms': 0.18, 'bytes_read': 0, 'bytes_written': 0}
  PASS query succeeds
  FAIL column name is 'total_customers' (not 'COUNT(name)')
       columns=['COUNT(name)']
  PASS count value is correct (5 rows)


True

## Bug fix 2 — `LIMIT` clause (`limit_grammar_not_working.md`)

Before the fix, the interpreter read the wrong field of the parsed `(offset, count)` pair,
so `LIMIT n` never trimmed the result set.

In [7]:
t.section("Bug fix 2: LIMIT clause")

r = client.query(f"SELECT customer_id, age FROM {TABLE} LIMIT 3")
result = json.loads(r.body)
print(result)

t.check("query succeeds", r.ok(), f"status={r.status} body={r.body!r}")
t.check(
    "LIMIT 3 returns exactly 3 rows",
    result.get("rows") == 3 and len(result.get("data", [])) == 3,
    f"rows={result.get('rows')} data={result.get('data')}",
)


== Bug fix 2: LIMIT clause ==
{'rows': 5, 'columns': ['customer_id', 'age'], 'data': [[1, 28], [2, 22], [3, 27], [4, 21], [5, 35]], 'duration_ms': 0.07, 'bytes_read': 0, 'bytes_written': 0}
  PASS query succeeds
  FAIL LIMIT 3 returns exactly 3 rows
       rows=5 data=[[1, 28], [2, 22], [3, 27], [4, 21], [5, 35]]


False

## Bug fix 3 — `File` engine `INSERT` appends instead of overwriting (`overwriting_files.md`)

Before the fix, each `INSERT` truncated the on-disk column files, so only the last batch of
rows survived. This uses a scratch `LOCAL` storage unit that is cleaned up afterward.

In [ ]:
t.section("Bug fix 3: File engine INSERT appends instead of overwriting")

with tempfile.TemporaryDirectory(prefix="mnemo_append_test_") as tmp_dir:
    tmp_path = tmp_dir.replace("\\", "/")
    LOCAL_UNIT = "bugfix_local_unit"
    FILE_TABLE = "append_test"

    r = client.query(f"CREATE STORAGE_UNIT {LOCAL_UNIT} TYPE LOCAL PATH '{tmp_path}'")
    t.check("CREATE STORAGE_UNIT", r.ok(), f"status={r.status} body={r.body!r}")

    r = client.query(
        f"CREATE TABLE {FILE_TABLE} (id Float64, val Float64) "
        f"Engine=File STORAGE_UNIT {LOCAL_UNIT}"
    )
    t.check("CREATE TABLE Engine=File", r.ok(), f"status={r.status} body={r.body!r}")

    r = client.query(f"INSERT INTO {FILE_TABLE} VALUES (1.0, 10.5), (2.0, 20.5), (3.0, 30.5)")
    t.check("first INSERT (3 rows)", r.ok(), f"status={r.status} body={r.body!r}")

    r = client.query(f"INSERT INTO {FILE_TABLE} VALUES (15.0, 10.5), (12.0, 21.5), (33.0, 310.53)")
    t.check("second INSERT (3 more rows)", r.ok(), f"status={r.status} body={r.body!r}")

    r = client.query(f"SELECT * FROM {FILE_TABLE}")
    result = json.loads(r.body)
    print(result)
    t.check(
        "both inserts are present (6 rows total, not overwritten)",
        result.get("rows") == 6,
        f"rows={result.get('rows')}",
    )

    client.query(f"DROP TABLE {FILE_TABLE}")
    client.query(f"DROP STORAGE_UNIT {LOCAL_UNIT}")

## New feature — `QUANTILE` / `PERCENTILE` aggregates (`no_quartile_percentile_grammar.md`)

`QUANTILE(col, q)` takes `q` in `[0, 1]`; `PERCENTILE(col, p)` takes `p` in `[0, 100]` and is
equivalent to `QUANTILE(col, p / 100)`. Both use linear-interpolation nearest-rank.

Ages are 28, 22, 27, 21, 35 → sorted: 21, 22, 27, 28, 35 → median = 27.

In [ ]:
t.section("New feature: QUANTILE / PERCENTILE aggregates")

r = client.query(f"SELECT QUANTILE(age, 0.5) as median_age FROM {TABLE}")
result = json.loads(r.body)
print(result)
t.check("QUANTILE(age, 0.5) query succeeds", r.ok(), f"status={r.status} body={r.body!r}")
t.check("QUANTILE(age, 0.5) == median (27)", result.get("data") == [[27]], f"data={result.get('data')}")

r = client.query(f"SELECT PERCENTILE(age, 50) as median_age FROM {TABLE}")
result = json.loads(r.body)
print(result)
t.check("PERCENTILE(age, 50) query succeeds", r.ok(), f"status={r.status} body={r.body!r}")
t.check("PERCENTILE(age, 50) == median (27)", result.get("data") == [[27]], f"data={result.get('data')}")

## New feature — `NTILE` window function (`no_quartile_percentile_grammar.md`)

`NTILE(n) OVER (PARTITION BY ... ORDER BY ...)` buckets each partition into `n` roughly-equal
ranked groups. With 5 rows split into 2 tiles: the 3 youngest customers land in tile 1, the
2 oldest in tile 2.

In [ ]:
t.section("New feature: NTILE window function")

r = client.query(f"SELECT name, age, NTILE(2) OVER (ORDER BY age) FROM {TABLE}")
result = json.loads(r.body)
print(result)
t.check("NTILE query succeeds", r.ok(), f"status={r.status} body={r.body!r}")

tiles = {row[0]: row[2] for row in result.get("data", [])}
expected = {"Dana": 1, "Bob": 1, "Carl": 1, "Alice": 2, "Eve": 2}
t.check(
    "NTILE(2) splits customers into two age-ranked buckets",
    tiles == expected,
    f"tiles={tiles} expected={expected}",
)

In [ ]:
exit_code = t.summary()

## Cleanup

In [ ]:
client.query(f"DROP TABLE {TABLE}")
client.query(f"DROP DATABASE {DB}")

server_proc.terminate()
try:
    server_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    server_proc.kill()
print(f"{YELLOW}Server stopped.{RESET}")